# Preprocessing

Imports and system path

In [6]:
import os
import sys

import numpy as np
import pandas as pd

# Go up THREE levels (project root directory)
project_root = os.path.dirname(os.path.dirname(os.getcwd()))
# Append the new path to sys.path
if project_root not in sys.path:
    sys.path.append(project_root)
    print("Project root added to sys.path")
else:
    print("Project root already in sys.path")

Project root already in sys.path


Processing `npt-HK4.gro` file into pandas dataframe

In [7]:
from utils import gro_processing as gp

# Data directory can be accessed due to root PATH we set previously
path = 'data/npt-HK4.gro'
abs_path = os.path.join(project_root, path)

# Extracts data from .gro file into multi-index DataFrame (unsorted)
df_gro, title, num_atoms, box_dimensions = gp.read_gro(abs_path, multiply=10, positions=True, velocities=False) # convert nm to Å  
    
# Checking
# df_gro
# molecules # display 

# Generating Molecule Meshes

Create `mol_meshes` dictionary, which contains 1501 individual molecule mesh. (i.e. `{1: mesh1, 2: mesh2, ... 1501: mesh1501}`)

In [ ]:
from utils.generate_mol_meshes import molecules_to_meshes

mol_meshes = molecules_to_meshes(df_gro, box_dimensions, num_processes=None, context='fork')
print(len(mol_meshes))        # 1501
print(mol_meshes[1])          # trimesh.Trimesh object

# Export, Import meshes

Users have two options to export meshes as `.npz` or `.ply` file.
- `.npz` is much smaller $\sim400\,\mathrm{MB}$, but exports longer $\sim50\,\mathrm{s}$
- `.ply` file is larger $\sim1.00\,\mathrm{GB}$, but exports shorter $\sim20\,\mathrm{s}$

In [3]:
# from utils.generate_mol_meshes import export_meshes

# # Export to either npz file or ply directory
# export_meshes(mol_meshes, path=path, export_format='ply', num_processes=None, context='fork')

Users have two options to import meshes from `.npz` or `.ply` file.
- `.npz` longer import $\sim20\,\mathrm{s}$ (parallel does not work well here) 
- `.ply` shorter import $\sim8\,\mathrm{s}$

In [4]:
from utils.generate_mol_meshes import import_meshes

# Import either npz files or ply directory (auto-detect)
# mol_meshes = import_meshes(path='npt-HK4_meshes.npz', num_processes=None, context='fork')
mol_meshes = import_meshes(path='npt-HK4_meshes', num_processes=None, context='fork')

Loading 1498 meshes with 8 cores: 100%|██████████| 1498/1498 [00:08<00:00, 178.95it/s]


# Neighbors Pairs (Blocking Algo)

All the functions have been merged into main function `find_neighbors`. User can export to `csv`, data that are exported include:
- centroids
- e-centroids
- neighbor_candidates
- neighbor_pairs

This function is designed to accommodate incomplete mol_meshes data. For example, if the following meshes are missing: `mol_id=[1,2,4]`, the output will simply exclude them (e.g., `{3:trimesh,5:trimesh,…,1501:trimesh}`). Consequently, the blocking algorithm continues uninterrupted using only the available meshes.

In [5]:
from utils.blocking_algo import find_neighbors

neighbor_pairs = find_neighbors(mol_meshes, df_gro, box_dimensions, path, k=10, export_csv=True)

Insert atom id for one molecule.
Use the following format: '20-30; 30; 20; 40-60' (for range use '-', for multi-input split using ';' 
----------------------------------------
You entered: [32-53]
----------------------------------------
Selected atom_id(s): [32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53]
----------------------------------------
Successfully exported npt-HK4_e_centroids.csv
Successfully exported npt-HK4_centroids.csv
Successfully exported npt-HK4_neighbor_candidates.csv
